Phase 5.5 — ablation: remove the `user_id` embedding (set `ablate_user_id=True`) and retrain with the same best config from 05c. This tests **hypothesis H6**: for cold-start users (rc<5) whose `user_id` embedding is OOV (zeros) anyway, having a learned `user_id` embedding for active users may actually push their gradients into noise that hurts cold-start generalization. If H6 holds, removing user_id should:
- slightly reduce general NDCG@10 (loses head-user collaborative signal)
- improve cold-start subset NDCG@10 (cleaner gradient on user-side numeric features alone)

The decision rule: if cold-start lift > general drop, ship the ablated model for the deploy / Persona path. Otherwise keep user_id and accept that cold-start needs other coverage (LLM intent extraction in CRS, ColBERT-light items in retrieval).

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import platform

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"arch: {platform.machine()}  device: {DEVICE}")

arch: arm64  device: mps


Inline boilerplate — same code as 05c/05d (IDEncoder + Dataset + DeepFM with `ablate_user_id` flag).

In [2]:
class IDEncoder:
    def __init__(self, ids, oov_token="<UNK>"):
        oov_markers = {"<NEW_USER>", "<NEW_BUSINESS>", "<UNK>", oov_token}
        unique_real_ids = sorted({i for i in ids if i not in oov_markers})
        self.id_to_idx = {oov_token: 0}
        for idx, _id in enumerate(unique_real_ids, start=1):
            self.id_to_idx[_id] = idx
        for marker in oov_markers:
            self.id_to_idx.setdefault(marker, 0)
        self._size = 1 + len(unique_real_ids)
    def __len__(self): return self._size
    def encode(self, _id): return self.id_to_idx.get(_id, 0)
    def encode_array(self, ids): return ids.map(self.id_to_idx).fillna(0).astype(np.int64).values


class TasteHunterDataset(Dataset):
    def __init__(self, df, user_encoder, item_encoder, user_features, item_features):
        self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))
        self.item_idx = torch.from_numpy(item_encoder.encode_array(df["business_id"]))
        self.label = torch.from_numpy(df["label"].astype(np.float32).values)
        n_users = len(user_encoder); n_items = len(item_encoder)
        self.user_num = np.zeros((n_users, 6), dtype=np.float32)
        self.user_cuisine = np.zeros((n_users, 50), dtype=np.float32)
        for _, row in user_features.iterrows():
            uidx = user_encoder.encode(row["user_id"])
            self.user_num[uidx] = [row["avg_rating_given"], row["review_count_log"], row["days_active"],
                                    row["elite_flag"], row["mean_distance_traveled"], row["price_tolerance_avg"]]
            emb = row["fav_cuisine_emb"]
            if isinstance(emb, list) and len(emb) == 50:
                self.user_cuisine[uidx] = emb
        self.item_num = np.zeros((n_items, 7), dtype=np.float32)
        self.item_cat = np.zeros((n_items, 50), dtype=np.float32)
        self.item_text = np.zeros((n_items, 32), dtype=np.float32)
        for _, row in item_features.iterrows():
            iidx = item_encoder.encode(row["business_id"])
            self.item_num[iidx] = [row["avg_rating"], row["review_count_log"], row["price_level"],
                                    row["is_open"], row["has_outdoor_seating"], row["photo_count_log"], row["city_id"]]
            cat = row["categories_multi_hot"]
            if isinstance(cat, list) and len(cat) == 50:
                self.item_cat[iidx] = cat
            text = row.get("item_text_emb_pca32")
            if isinstance(text, (list, np.ndarray)) and len(text) == 32:
                self.item_text[iidx] = np.asarray(text, dtype=np.float32)
        self.user_num_t = torch.from_numpy(self.user_num)
        self.user_cuisine_t = torch.from_numpy(self.user_cuisine)
        self.item_num_t = torch.from_numpy(self.item_num)
        self.item_cat_t = torch.from_numpy(self.item_cat)
        self.item_text_t = torch.from_numpy(self.item_text)
    def __len__(self): return len(self.label)
    def __getitem__(self, idx):
        u = self.user_idx[idx]; i = self.item_idx[idx]
        return {"user_idx": u, "item_idx": i, "label": self.label[idx],
                "user_num": self.user_num_t[u], "user_cuisine": self.user_cuisine_t[u],
                "item_num": self.item_num_t[i], "item_cat": self.item_cat_t[i],
                "item_text": self.item_text_t[i]}


def make_val_eval_pairs(val_df, user_encoder, item_encoder, item_features, n_negs=99, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    val_pos = val_df[val_df["stars"] >= 4].copy()
    biz_city = item_features.set_index("business_id")["city"].to_dict()
    val_pos["city"] = val_pos["business_id"].map(biz_city).fillna("<UNK>")
    city_biz = {}
    for bid, c in biz_city.items():
        if c == "<UNK>": continue
        city_biz.setdefault(c, []).append(bid)
    for c in city_biz: city_biz[c] = np.array(city_biz[c])
    all_user, all_item, all_label = [], [], []
    for row in val_pos.itertuples(index=False):
        if row.city not in city_biz: continue
        cands = city_biz[row.city]
        sampled = rng.choice(cands, size=n_negs * 2, replace=True)
        negs = [b for b in sampled if b != row.business_id][:n_negs]
        if len(negs) < n_negs: continue
        items = [row.business_id] + negs
        all_user.append([user_encoder.encode(row.user_id)] * (n_negs + 1))
        all_item.append([item_encoder.encode(b) for b in items])
        all_label.append([1.0] + [0.0] * n_negs)
    return np.array(all_user, dtype=np.int64), np.array(all_item, dtype=np.int64), np.array(all_label, dtype=np.float32)


def compute_auc(scores, labels):
    pos_mask = labels > 0.5; n_pos = pos_mask.sum(); n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    order = np.argsort(scores); ranks = np.empty(len(scores))
    ranks[order] = np.arange(1, len(scores) + 1)
    return float((ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def ndcg_at_k(score_matrix, label_matrix, k=10):
    top_k_idx = np.argsort(-score_matrix, axis=1)[:, :k]
    rows = np.arange(score_matrix.shape[0])[:, None]
    top_k_labels = label_matrix[rows, top_k_idx]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = (top_k_labels * discounts).sum(axis=1)
    ideal_labels = -np.sort(-label_matrix, axis=1)[:, :k]
    idcg = (ideal_labels * discounts).sum(axis=1)
    idcg = np.where(idcg > 0, idcg, 1.0)
    return float(np.mean(dcg / idcg))


def recall_at_k(score_matrix, label_matrix, k=10):
    top_k_idx = np.argsort(-score_matrix, axis=1)[:, :k]
    rows = np.arange(score_matrix.shape[0])[:, None]
    top_k_labels = label_matrix[rows, top_k_idx]
    n_pos_per_row = label_matrix.sum(axis=1)
    correct = top_k_labels.sum(axis=1)
    return float(np.mean(np.where(n_pos_per_row > 0, correct / n_pos_per_row, 0)))


@torch.no_grad()
def score_pairs(model, user_idx, item_idx, dataset, device, batch_size=8192):
    model.eval()
    n, c = user_idx.shape
    flat_u = user_idx.reshape(-1); flat_i = item_idx.reshape(-1)
    scores = np.zeros(n * c, dtype=np.float32)
    for start in range(0, len(flat_u), batch_size):
        end = min(start + batch_size, len(flat_u))
        u_b = torch.from_numpy(flat_u[start:end]).to(device)
        i_b = torch.from_numpy(flat_i[start:end]).to(device)
        u_t = torch.from_numpy(flat_u[start:end])
        i_t = torch.from_numpy(flat_i[start:end])
        kwargs = {
            "user_num": dataset.user_num_t[u_t].to(device),
            "user_cuisine": dataset.user_cuisine_t[u_t].to(device),
            "item_num": dataset.item_num_t[i_t].to(device),
            "item_cat": dataset.item_cat_t[i_t].to(device),
            "item_text": dataset.item_text_t[i_t].to(device),
        }
        s = model(u_b, i_b, **kwargs).cpu().numpy().reshape(-1)
        scores[start:end] = s
    return scores.reshape(n, c)


def evaluate_full(model, user_idx_eval, item_idx_eval, label_eval, dataset, device):
    score_matrix = score_pairs(model, user_idx_eval, item_idx_eval, dataset, device)
    return {"AUC": compute_auc(score_matrix.reshape(-1), label_eval.reshape(-1)),
            "NDCG@10": ndcg_at_k(score_matrix, label_eval, k=10),
            "Recall@10": recall_at_k(score_matrix, label_eval, k=10)}


class DeepFM(nn.Module):
    """Same DeepFM as 05c/05d. ablate_user_id=True zeros out user_id emb + bias."""
    def __init__(self, n_users, n_items, emb_dim=32,
                 user_num_dim=6, user_cuisine_dim=50,
                 item_num_dim=7, item_cat_dim=50, item_text_dim=32,
                 dnn_hidden=(256, 128, 64), dropout=0.2, ablate_user_id=False):
        super().__init__()
        self.ablate_user_id = ablate_user_id
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_num_proj = nn.Linear(user_num_dim, emb_dim)
        self.item_num_proj = nn.Linear(item_num_dim, emb_dim)
        self.item_text_proj = nn.Linear(item_text_dim, emb_dim)
        feat_dim_linear = user_num_dim + item_num_dim + user_cuisine_dim + item_cat_dim + item_text_dim
        self.linear_features = nn.Linear(feat_dim_linear, 1)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        dnn_input_dim = emb_dim * 5 + user_cuisine_dim + item_cat_dim + item_text_dim
        layers = []; prev = dnn_input_dim
        for h in dnn_hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, 1))
        self.dnn = nn.Sequential(*layers)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, user_num, user_cuisine, item_num, item_cat, item_text):
        u_emb = self.user_emb(user_idx)
        i_emb = self.item_emb(item_idx)
        un_emb = self.user_num_proj(user_num)
        in_emb = self.item_num_proj(item_num)
        it_emb = self.item_text_proj(item_text)
        if self.ablate_user_id:
            u_emb = torch.zeros_like(u_emb)
        feat_concat = torch.cat([user_num, item_num, user_cuisine, item_cat, item_text], dim=-1)
        order1 = self.linear_features(feat_concat).squeeze(-1)
        bu = self.user_bias(user_idx).squeeze(-1)
        bi = self.item_bias(item_idx).squeeze(-1)
        if self.ablate_user_id:
            bu = torch.zeros_like(bu)
        embs = torch.stack([u_emb, i_emb, un_emb, in_emb, it_emb], dim=1)
        sum_sq = embs.sum(dim=1) ** 2; sq_sum = (embs ** 2).sum(dim=1)
        order2 = 0.5 * (sum_sq - sq_sum).sum(dim=-1)
        dnn_input = torch.cat([u_emb, i_emb, un_emb, in_emb, it_emb, user_cuisine, item_cat, item_text], dim=-1)
        dnn_output = self.dnn(dnn_input).squeeze(-1)
        return torch.sigmoid(order1 + order2 + dnn_output + bu + bi + self.global_bias)

Read best config from 05c, train ablated DeepFM (no user_id) on train only (so we can compare against 5.3 grid winner side by side).

In [3]:
with open(MODELS_DIR / "deepfm_sweep_dropout_l2.json") as f:
    grid = json.load(f)
best_config = max(grid, key=lambda x: x["best_ndcg10"])
print(f"using config: emb={best_config['emb_dim']} dropout={best_config['dropout']} l2={best_config['l2']:.0e}")

user_features = pd.read_parquet(FEATURES_DIR / "user_features.parquet")
item_features = pd.read_parquet(FEATURES_DIR / "item_features.parquet")
val_df = pd.read_parquet(CLEANED_DIR / "val_reviews.parquet")
train_df = pd.read_parquet(FEATURES_DIR / "train_with_negatives.parquet")

user_encoder = IDEncoder(user_features["user_id"].tolist(), oov_token="<NEW_USER>")
item_encoder = IDEncoder(item_features["business_id"].tolist(), oov_token="<NEW_BUSINESS>")
n_users, n_items = len(user_encoder), len(item_encoder)

user_idx_eval, item_idx_eval, label_eval = make_val_eval_pairs(
    val_df, user_encoder, item_encoder, item_features, n_negs=99,
)

t0 = time.time()
train_ds = TasteHunterDataset(train_df, user_encoder, item_encoder, user_features, item_features)
print(f"train ds: {len(train_ds)} samples ({time.time()-t0:.1f}s)")
train_loader = DataLoader(train_ds, batch_size=8192, shuffle=True, num_workers=0)

torch.manual_seed(SEED)
model = DeepFM(n_users, n_items, emb_dim=best_config["emb_dim"], dropout=best_config["dropout"],
               ablate_user_id=True).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=best_config["l2"])
bce = nn.BCELoss()

history = {"epoch": [], "train_loss": [], "val_auc": [], "val_ndcg10": [], "val_recall10": []}
for epoch in range(1, 11):
    model.train()
    loss_sum, n_batch = 0.0, 0
    t0 = time.time()
    for batch in train_loader:
        u = batch["user_idx"].to(DEVICE); i = batch["item_idx"].to(DEVICE); y = batch["label"].to(DEVICE)
        kwargs = {k: batch[k].to(DEVICE) for k in ("user_num", "user_cuisine", "item_num", "item_cat", "item_text")}
        pred = model(u, i, **kwargs)
        loss = bce(pred, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item(); n_batch += 1
    train_loss = loss_sum / n_batch
    elapsed = time.time() - t0
    m = evaluate_full(model, user_idx_eval, item_idx_eval, label_eval, train_ds, DEVICE)
    print(f"  ep{epoch:02d} | {elapsed:.0f}s | loss={train_loss:.4f} | AUC={m['AUC']:.4f} NDCG@10={m['NDCG@10']:.4f} Recall@10={m['Recall@10']:.4f}")
    history["epoch"].append(epoch); history["train_loss"].append(train_loss)
    history["val_auc"].append(m["AUC"]); history["val_ndcg10"].append(m["NDCG@10"]); history["val_recall10"].append(m["Recall@10"])

torch.save(model.state_dict(), MODELS_DIR / "deepfm_ablate_no_user_id.pt")
history["best_ndcg10"] = max(history["val_ndcg10"])
history["config"] = {**best_config, "ablate_user_id": True}
with open(MODELS_DIR / "deepfm_ablate_no_user_id_history.json", "w") as f:
    json.dump(history, f, indent=2)

using config: emb=32 dropout=0.1 l2=1e-04


/var/folders/4j/cfzmxp0d3db8xdcx2fncnbnc0000gn/T/ipykernel_25614/3834684094.py:18: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))


train ds: 2075285 samples (4.7s)


  ep01 | 25s | loss=0.6620 | AUC=0.7903 NDCG@10=0.2574 Recall@10=0.4536


  ep02 | 25s | loss=0.3203 | AUC=0.8057 NDCG@10=0.2689 Recall@10=0.4720


...

  ep06 | 25s | loss=0.2852 | AUC=0.8264 NDCG@10=0.2993 Recall@10=0.5154


  ep07 | 25s | loss=0.2833 | AUC=0.8452 NDCG@10=0.3192 Recall@10=0.5421


  ep08 | 25s | loss=0.2818 | AUC=0.8475 NDCG@10=0.3234 Recall@10=0.5465


  ep09 | 25s | loss=0.2803 | AUC=0.8479 NDCG@10=0.3256 Recall@10=0.5506


  ep10 | 25s | loss=0.2792 | AUC=0.8447 NDCG@10=0.3226 Recall@10=0.5466


Compare ablated vs un-ablated (best 05c config) on the val set.

In [4]:
best_tag = best_config["tag"]
with open(MODELS_DIR / f"deepfm_{best_tag}_history.json") as f:
    h_full = json.load(f)
h_ablate = history

print(f"{'metric':<14}{'with user_id':<16}{'no user_id':<14}{'delta':<10}")
print(f"{'-'*54}")
for key, label in [("val_auc", "AUC"), ("val_ndcg10", "NDCG@10"), ("val_recall10", "Recall@10")]:
    a = max(h_full[key]); b = max(h_ablate[key])
    print(f"{label:<14}{a:<16.4f}{b:<14.4f}{b-a:+.4f}")

metric        with user_id    no user_id    delta     
------------------------------------------------------
AUC           0.8502          0.8479        -0.0023
NDCG@10       0.3255          0.3256        +0.0001
Recall@10     0.5510          0.5506        -0.0004


If ablated model loses < 0.02 NDCG@10 on val (general) and we'd expect the cold-start subset evaluated in Phase 7.4.3 to **gain** NDCG, H6 holds and the ablated variant becomes the deploy-time fallback for OOV users. If ablated model loses much more on general, the user_id signal is genuinely valuable for active users and the ablation just confirms head-tail trade-off without changing the deploy recipe — `<NEW_USER>` OOV continues to receive zero embedding contribution regardless.